# Day 2 - Local Study Notes Generator

Give it any URL and it will generate structured study notes using a local Ollama model — no API costs, data stays on your machine!

**Output includes:**
- Summary of the topic
- Key Concepts
- Quiz Questions with Answers

**Note:** Works best on static pages (Wikipedia, docs, blogs). For JavaScript-heavy sites, use a Selenium-based scraper.

**Model:** `llama3.2` (switch to `llama3.2:1b` if your machine is slow)

In [1]:
# Imports

import os
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
# Ollama client setup

OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [3]:
# Website scraper — fetches and cleans webpage text

def fetch_website_contents(url):
    headers = {"User-Agent": "Mozilla/5.0 (compatible; StudyNotesBot/1.0)"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    if soup.body is None:
        raise ValueError(f"Could not parse page body from {url}. Status: {response.status_code}")
    for tag in soup.body.find_all(["script", "style", "img", "input"]):
        tag.decompose()
    return soup.body.get_text(separator="\n", strip=True)

In [4]:
# System prompt — tells the model what to generate

system_prompt = """
You are an expert educator and study notes generator.
Given the contents of a webpage, your job is to produce structured study notes in Markdown.

Your notes must include:
1. **Summary** - A brief overview of the topic
2. **Key Concepts** - The most important ideas as bullet points
3. **Quiz Questions** - 3 to 5 questions to test understanding
4. **Answers** - After all questions, provide answers with a brief explanation for each

Keep the language clear and beginner friendly.
Do not include navigation text, ads, or irrelevant content.
Respond only in Markdown.
"""

In [5]:
# Build messages list for the model

MAX_CHARS = 6000  # keep well within llama3.2's context window

def messages_for(text):
    truncated = text[:MAX_CHARS] + ("\n\n[Content truncated for length]" if len(text) > MAX_CHARS else "")
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": (
            "Here is the content of a webpage. "
            "Generate structured study notes with ALL sections: Summary, Key Concepts, Quiz Questions, and Answers.\n\n"
            + truncated
        )}
    ]

In [6]:
# Main function — ties everything together

def generate_study_notes(url):
    text = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(text)
    )
    return response.choices[0].message.content

In [7]:
# Display function — renders the notes as formatted Markdown

def display_study_notes(url):
    notes = generate_study_notes(url)
    display(Markdown(notes))

In [8]:
display_study_notes("https://en.wikipedia.org/wiki/Odisha")

**Summary**
===

Odisha, formerly known as Orissa, is a state located in eastern India. It is the eighth-largest state by area and the eleventh-largest by population with over 41 million inhabitants.

**Key Concepts**

*   Geography: Odisha has a coastline of 485 kilometers along the Bay of Bengal
*   Economy: The state's economy is driven by agriculture, particularly rice, wheat, and soybeans
*   Culture: Odisha is known for its rich cultural heritage, including its traditional dance forms such as OdISSi\~\-L~a~sh~na and folk music
*   Government: The state has a unicameral legislature and is governed by the Government of Odisha.

**Quiz Questions**

1.  What is the official language of Odisha?
    *   A) Hindi
    *   B) Odia
    *   C) English
2.  Which of the following is a major industry in Odisha?
    *   A) Textiles
    *   B) Agriculture
    *   C) Steel
3.  What is the capital city of Odisha?
    *   A) Bhubaneswar
    *   B) Cuttack
    *   C) Puri

**Answers**

1.  B) Odia
    *   Explanation: Odisha uses both the Odia language and English as official languages.
2.  B) Agriculture
    *   explanation: agriculture is a major component of Odisha&#x27;s economy with crop farming accounting for around 40% of the state production.
3.  A) Bhubaneswar
    *   Explanation: The capital was originally located in Cuttack but it had to be moved during political unrest and ultimately settled upon Bhubaneswar